### Load the Dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups

dataset = fetch_20newsgroups(subset="train")

documents = dataset.data
labels = dataset.target
categories = dataset.target_names

In [5]:
documents = documents[:300]
labels = labels[:300]

### Preprocessing

In [11]:
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [13]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [14]:
def preprocess(text):

    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [15]:
processed_docs = [preprocess(doc) for doc in documents]

### TF-IDF Vectorization

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(processed_docs)

### Query Preprocessing

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=3):

    query = preprocess(query)
    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    ranked = scores.argsort()[::-1][:top_k]
    

    for i in ranked:
        print("Score:", scores[i])
        print(documents[i][:500])
        print()

In [30]:
search("married")

Score: 0.11446645780773768
From: cramer@optilink.COM (Clayton Cramer)
Subject: Re: New Study Out On Gay Percentage
Organization: Optilink Corporation, Petaluma, CA
Lines: 19

In article <C5L0v1.JCv@news.cso.uiuc.edu>, dans@uxa.cso.uiuc.edu (Dan S.) writes:
> Don't forget about the culture.  Sadly, we don't (as a society) look upon
> homosexuality as normal (and as we are all too well aware, there are alot
> of people who condemn it).  As a result, the gay population is not encouraged
> to develop "non-promiscuous" relatio

Score: 0.06863158947052587
From: cramer@optilink.COM (Clayton Cramer)
Subject: Re: New Study Out On Gay Percentage
Organization: Optilink Corporation, Petaluma, CA
Lines: 31

In article <1993Apr16.164638.27218@galileo.cc.rochester.edu>, as010b@uhura.cc.rochester.edu (Tree of Schnopia) writes:
> In <15378@optilink.COM> cramer@optilink.COM (Clayton Cramer) writes:
# #The article also contains numbers on the number of sexual partners.
# #The median number of sexual part

## Using SentenceTrnasformer for sentence embedding

In [73]:
from sklearn.datasets import fetch_20newsgroups

dataset = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

documents = dataset.data

In [74]:
documents = documents[:5000]
labels = dataset.target[:5000]
categories = dataset.target_names

In [76]:
import re

def clean_text(text):

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [77]:
clean_docs = [clean_text(doc) for doc in documents]

In [42]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [78]:
doc_embeddings = model.encode(documents)

In [48]:
def search(query, top_k=2):

    query_embedding = model.encode([query])

    scores = cosine_similarity(query_embedding, doc_embeddings).flatten()
    ranked = scores.argsort()[::-1][:top_k]

    for i in ranked:
        print(scores[i])
        print(documents[i][:200])
        print()

In [83]:
search('wealth')

0.44274265
: significantly less than the value of many automobiles.  And for those who will
: argue that the animals out there stealing cars and everything else (not to
: mention committing COMPLETELY senseless 

0.42572635

Ah and how...??? Amen to that one!!!!!!  Thanks Chuck for sharing...
after all, no one can serve two masters...God and money......
after all, the preciousness of God as Lord and Savior is far more va

